<a href="https://colab.research.google.com/github/AyaAbdElNaem/Deep_Learning/blob/main/Task_Week3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install git+https://github.com/tensorflow/docs

  Cloning https://github.com/tensorflow/docs to /tmp/pip-req-build-dvezvpa3
  Running command git clone --filter=blob:none --quiet https://github.com/tensorflow/docs /tmp/pip-req-build-dvezvpa3
  Resolved https://github.com/tensorflow/docs to commit 0054afff57cd4a4ea5389088a89942603461ee6f
  Preparing metadata (setup.py) ... done
  Created wheel for tensorflow-docs: filename=tensorflow_docs-2026.5.12.61975-py3-none-any.whl size=186932 sha256=cbe69aa5c1fba7f62e70c902e44834898095a514b53ff7104d3f9985605cda55
  Stored in directory: /tmp/pip-ephem-wheel-cache-c0qhhek1/wheels/3e/88/34/48d2789bc9d37b33ddce06bccc454fae0285e5396d0a5be9d9
Successfully built tensorflow-docs


In [31]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# tensorflow libraries
import tensorflow as tf
from tensorflow import keras
import tensorflow_docs as tfdocs
from tensorflow.keras import layers

In [32]:
data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep=r"\s+", skiprows=22, header=None)
data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
target = raw_df.values[1::2, 2]

# Create a dictionary-like object to mimic the original boston dataset structure
# This is done to maintain compatibility with downstream code that might expect 'data' and 'target' attributes
boston = type('BostonDataset', (object,), {})
boston.data = data
boston.target = target
boston.feature_names = [
    'CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT'
]
boston.DESCR = """Boston house prices dataset (fetched from original source)"""

features = np.array(boston.data)
target = np.array(boston.target)

In [33]:
n_training_samples = features.shape[0]
n_dim = features.shape[1]

print('The dataset has', n_training_samples, 'training samples.')
print('The dataset has', n_dim, 'features.')

The dataset has 506 training samples.
The dataset has 13 features.


In [34]:
def normalize(dataset):
    mu = np.mean(dataset, axis = 0)
    sigma = np.std(dataset, axis = 0)
    return (dataset - mu)/sigma

In [35]:
features_norm = normalize(features)

In [36]:
np.random.seed(42)
rnd = np.random.rand(len(features_norm)) < 0.8

train_x = features_norm[rnd]
train_y = target[rnd]
dev_x = features_norm[~rnd]
dev_y = target[~rnd]

print(train_x.shape)
print(train_y.shape)
print(dev_x.shape)
print(dev_y.shape)

(399, 13)
(399,)
(107, 13)
(107,)


Try to determine which architecture (number of layers and number of neurons) is not overfitting the Boston dataset. When the network starts overfitting? Which network would give a good result?

In [37]:
def build_model(opt):
  # create model
	model = keras.Sequential()
	model.add(layers.Dense(15, input_dim = 784, activation = 'relu')) # add first hidden layer and set input dimensions
	model.add(layers.Dense(10, activation = 'softmax')) # add output layer
	# compile model
	model.compile(loss = 'categorical_crossentropy', optimizer = opt, metrics = ['categorical_accuracy'])
	return model

In [21]:
# from keras.models import Sequential
# from keras.layers import Dense, Input # 1. قم باستيراد Input

# # الطريقة الحديثة والصحيحة
# model = Sequential([
#     Input(shape=(10,)), # 2. اجعلها الطبقة الأولى لتحديد أبعاد المدخلات
#     Dense(64, activation='relu'), # 3. امسح input_shape من هنا
#     Dense(1)
# ])

In [22]:
model = build_model(tf.keras.optimizers.SGD(momentum = 0.0, learning_rate = 0.01))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 15)             │        11,775 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           160 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,935 (46.62 KB)

 Trainable params: 11,935 (46.62 KB)

 Non-trainable params: 0 (0.00 B)

In [38]:
EPOCHS = 1000

history = model.fit(
  train_x, train_y,
  epochs = EPOCHS, verbose = 0,
  batch_size = train_x.shape[0],
  callbacks = [tfdocs.modeling.EpochDots()])

ValueError: Exception encountered when calling Sequential.call().

[1mInput 0 with name 'None' of layer 'dense_2' is incompatible with the layer: expected axis -1 of input shape to have value 784, but received input with shape (399, 13)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(399, 13), dtype=float32)
  • training=True
  • mask=None
  • kwargs=<class 'inspect._empty'>

In [39]:
hist = pd.DataFrame(history.history)
hist['epoch'] = history.epoch
hist.tail()

NameError: name 'history' is not defined

In [40]:
test_loss, test_accuracy = model.evaluate(data_test_norm, labels_test, verbose = 0)
print('The accuracy on the test set is equal to: ', int(test_accuracy*100), '%.')

NameError: name 'data_test_norm' is not defined